# Procesamiento Experimental (video l2.mp4)

Este cuaderno procesa el video del cohete y construye su trayectoria.

**Datos del ensayo:** botella de 1.5 L, fracción de agua φ = 1/3 (0.5 L), inflada con bombín de bicicleta (la presión no quedó registrada).

**Geometría del montaje (l2):** lanzador de 1.74 m; cámara a 12 m de distancia, a 0 m de altura (a ras de suelo).

**Antes de ejecutar:** sube `l2.mp4` a la misma carpeta que este cuaderno (en Colab: panel de archivos de la izquierda).

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### 1.1 Lectura del video

In [ ]:
VIDEO = "l2.mp4"

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS)
ANCHO = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
ALTO = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print(f"Cuadros por segundo: {FPS:.3f}")
print(f"Resolucion: {ANCHO} x {ALTO} pixeles")


def obtener_frame(t):
    """Devuelve el cuadro del video en el instante t (s) como imagen RGB."""
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, round(t * FPS))
    ok, frame_bgr = cap.read()
    cap.release()
    if not ok:
        raise RuntimeError(f"No se pudo leer el video en t = {t} s")
    return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

### 1.1 Explorador de cuadros

Revisión cuadro por cuadro.

In [ ]:
def ver_cuadros(t_inicio, t_fin, paso=0.1, columnas=5, con_grilla=True):
    """Muestra en una grilla los cuadros entre t_inicio y t_fin, uno cada 'paso' segundos."""
    tiempos = np.arange(t_inicio, t_fin, paso)
    filas = int(np.ceil(len(tiempos) / columnas))
    fig, axs = plt.subplots(filas, columnas, figsize=(3 * columnas, 5.3 * filas))
    axs = np.array(axs).reshape(-1)

    for ax, t in zip(axs, tiempos):
        ax.imshow(obtener_frame(t))
        if con_grilla:
            for x in range(0, ANCHO, 100):
                ax.axvline(x, color="red", lw=0.3, alpha=0.5)
            for y in range(0, ALTO, 100):
                ax.axhline(y, color="lime", lw=0.3, alpha=0.5)
        ax.set_title(f"t = {t:.3f} s", fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axs[len(tiempos):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

ver_cuadros(18, 22, paso=1 / FPS)

### 1.2 Calibración de escala: altura de una persona

El operador, de pie en t = 0 s, mide 1.71 m.

In [ ]:
frame_persona = obtener_frame(0)

cabeza_px = 362   # fila del tope del casco
pies_px = 472     # fila donde el zapato toca el pasto
altura_px = pies_px - cabeza_px
altura_m = 1.71

escala = altura_m / altura_px   # metros por pixel
print(f"Altura de la persona: {altura_px} px")
print(f"Escala: {escala:.5f} m/px")

fig, ax = plt.subplots(figsize=(3.5, 5))
ax.imshow(frame_persona)
ax.axhline(cabeza_px, color="red", lw=1.5, label="cabeza")
ax.axhline(pies_px, color="lime", lw=1.5, label="pies")
ax.set_xlim(180, 330)
ax.set_ylim(500, 320)
ax.legend(fontsize=8, loc="upper right")
ax.set_title(f"Calibracion de escala (t = 0 s)\naltura = {altura_px} px")
plt.tight_layout()
plt.show()

### 1.3 Posiciones del cohete (detección automática)

In [ ]:
def candidatos_blancos(frame_rgb, caja, s_max=60, v_min=175):
    """Manchas blancas o brillantes y compactas dentro de la caja (x0, x1, y0, y1).

    Devuelve una lista de (columna, fila, area) en coordenadas de la imagen completa.
    """
    x0, x1, y0, y1 = caja
    recorte_bgr = cv2.cvtColor(frame_rgb[y0:y1, x0:x1], cv2.COLOR_RGB2BGR)
    hsv = cv2.cvtColor(recorte_bgr, cv2.COLOR_BGR2HSV)
    saturacion = hsv[:, :, 1]
    brillo = hsv[:, :, 2]
    mascara = ((saturacion < s_max) & (brillo >= v_min)).astype(np.uint8) * 255

    n_manchas, _, stats, centroides = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    candidatos = []
    for k in range(1, n_manchas):
        x, y, w, h, area = stats[k]
        if area < 4:
            continue
        if area / (w * h) < 0.15:   # descarta manchas muy ralas
            continue
        candidatos.append((centroides[k][0] + x0, centroides[k][1] + y0, area))
    return candidatos


def trackear(caja, t_visibles, semilla_xy, semilla_v, s_max=60, v_min=175):
    """Sigue el cohete cuadro a cuadro: elige la mancha mas cercana a la posicion predicha.

    semilla_xy: (columna, fila) inicial en px; semilla_v: velocidad inicial (px/s).
    Devuelve (columnas, filas) en px.
    """
    dt = 1.0 / FPS
    x_prev, y_prev = semilla_xy
    vx_prev, vy_prev = semilla_v
    col_px, fila_px = [x_prev], [y_prev]

    for t in t_visibles[1:]:
        candidatos = candidatos_blancos(obtener_frame(t), caja, s_max=s_max, v_min=v_min)
        col_pred = x_prev + vx_prev * dt
        fila_pred = y_prev + vy_prev * dt
        distancias = [(c[0] - col_pred) ** 2 + (c[1] - fila_pred) ** 2 for c in candidatos]
        mejor = candidatos[int(np.argmin(distancias))]
        vx_prev = (mejor[0] - x_prev) / dt
        vy_prev = (mejor[1] - y_prev) / dt
        x_prev, y_prev = mejor[0], mejor[1]
        col_px.append(x_prev)
        fila_px.append(y_prev)

    return np.array(col_px), np.array(fila_px)

In [ ]:
# Posiciones obtenidas con el detector de arriba.
# Cada fila: (t en segundos, columna_px, fila_px, dy_camara_px)
# dy_camara: cuanto bajo la escena respecto a t = 21.4 s (la camara estaba en la mano)
datos = [
    (19.026, 229.2, 433.5, -5.17), (19.059, 241.5, 409.8, -5.19), (19.092, 269.8, 351.5, -5.19),
    (19.126, 279.6, 330.7, -5.18), (19.159, 305.9, 272.5, -5.14), (19.193, 321.2, 229.5, -5.10),
    (19.226, 333.1, 200.6, -5.11), (19.260, 339.8, 175.8, -5.09), (19.293, 343.4, 164.2, -5.09),
    (19.326, 348.8, 140.0, -5.10), (19.360, 351.0, 123.2, -5.09), (19.393, 356.8, 106.4, -5.06),
    (19.427, 359.6,  89.0, -5.07), (19.460, 361.5,  76.2, -5.06), (19.494, 361.0,  67.7, -5.05),
    (19.527, 366.2,  58.7, -4.90), (19.560, 370.5,  48.3, -4.89), (19.594, 370.7,  47.5, -4.78),
    (19.627, 376.8,  33.9, -4.76), (19.661, 377.1,  26.4, -4.74), (19.694, 376.9,  21.1, -4.73),
    (19.728, 377.5,  15.6, -4.81), (19.761, 378.8,  11.0, -4.80), (19.795, 380.8,  11.7, -5.10),
    (19.828, 380.5,   7.5, -5.22), (19.861, 379.2,   7.6, -3.61), (19.895, 376.4,  11.3, -1.57),
    (19.928, 379.1,   8.3, -0.36), (19.962, 379.1,   9.3,  0.73), (19.995, 380.7,  10.8,  0.14),
    (20.029, 383.6,  11.1, -0.33), (20.062, 383.7,  12.5,  0.08), (20.095, 386.1,  13.2,  0.08),
    (20.129, 385.0,  16.7,  0.07), (20.162, 387.1,  17.7,  0.13), (20.196, 386.7,  20.9, -0.74),
    (20.229, 389.5,  27.0, -0.81), (20.263, 392.9,  31.1, -1.43), (20.296, 390.5,  35.0, -1.17),
    (20.330, 390.0,  41.2, -1.33), (20.363, 390.7,  46.7, -1.15), (20.396, 391.5,  53.2, -0.93),
    (20.430, 393.6,  58.2, -1.06), (20.463, 395.0,  64.3, -0.83), (20.497, 396.4,  69.3, -0.83),
    (20.530, 399.9,  78.8, -0.91), (20.564, 402.3,  85.0, -0.94), (20.597, 403.5,  90.4, -0.86),
    (20.630, 403.8,  97.5, -0.27), (20.664, 404.3, 104.4,  0.49), (20.697, 404.5, 111.9,  0.60),
    (20.731, 404.0, 120.3,  0.83), (20.764, 403.6, 130.9,  0.78), (20.798, 404.3, 140.9,  1.07),
    (20.831, 405.9, 150.6,  1.05), (20.865, 408.8, 159.6,  0.94), (20.898, 408.3, 168.6,  0.83),
    (20.931, 406.5, 179.0,  0.89), (20.965, 408.4, 186.9,  0.64), (20.998, 411.1, 193.8,  0.43),
    (21.032, 411.9, 202.9,  0.00), (21.065, 413.2, 211.6,  0.02), (21.099, 414.5, 220.1,  0.10),
    (21.132, 415.5, 228.9, -0.03), (21.165, 416.6, 239.0,  0.06), (21.199, 418.7, 249.5,  0.02),
    (21.232, 420.0, 258.5, -0.01), (21.266, 422.8, 267.5, -0.10), (21.299, 425.5, 275.4, -0.10),
    (21.333, 427.1, 284.0, -0.11), (21.366, 428.5, 292.5,  0.00), (21.400, 429.7, 301.8,  0.00),
    (21.433, 430.1, 309.2, -0.14), (21.466, 425.0, 320.6, -0.31), (21.500, 429.7, 335.4, -0.44),
    (21.533, 432.1, 348.4, -0.62), (21.567, 432.5, 359.0, -0.68), (21.600, 433.8, 370.2, -0.97),
    (21.634, 435.1, 380.1, -1.06), (21.667, 435.9, 390.4, -1.18), (21.700, 436.6, 399.4, -1.30),
    (21.734, 436.2, 413.3, -1.51), (21.767, 437.2, 422.0, -1.63), (21.801, 437.5, 433.0, -1.77),
    (21.834, 437.5, 444.8, -1.84), (21.868, 437.3, 456.8, -1.52), (21.901, 437.0, 467.9, -1.48),
]

t_visibles = np.array([d[0] for d in datos])
col_px = np.array([d[1] for d in datos], float)
fila_px = np.array([d[2] for d in datos], float)
dy_cam = np.array([d[3] for d in datos], float)

print(f"Puntos medidos: {len(datos)}")
print(f"Tiempo: de {t_visibles[0]:.3f} s a {t_visibles[-1]:.3f} s")

In [ ]:
# Detector aplicado a los mismos instantes de 'datos'.
# La posicion y velocidad iniciales se toman de sus dos primeros puntos.
def imprimir_datos(t_visibles, col_px, fila_px, dy_cam, por_linea=3):
    "Imprime las posiciones en el formato de la lista 'datos'."
    filas = [f"({t:.3f}, {c:5.1f}, {f:5.1f}, {d:5.2f})" for t, c, f, d in zip(t_visibles, col_px, fila_px, dy_cam)]
    print("datos = [")
    for i in range(0, len(filas), por_linea):
        print("    " + ", ".join(filas[i:i + por_linea]) + ",")
    print("]")


caja = (0, ANCHO, 0, ALTO)   # zona de busqueda: el cuadro completo

dt = t_visibles[1] - t_visibles[0]
semilla_xy = (col_px[0], fila_px[0])
semilla_v = ((col_px[1] - col_px[0]) / dt, (fila_px[1] - fila_px[0]) / dt)

col_det, fila_det = trackear(caja, t_visibles, semilla_xy, semilla_v)

imprimir_datos(t_visibles, col_det, fila_det, dy_cam)
print(f"\nDiferencia maxima con 'datos': {np.abs(col_det - col_px).max():.1f} px en columna, "
      f"{np.abs(fila_det - fila_px).max():.1f} px en fila")

**Verificación visual.**

In [ ]:
filas_g = int(np.ceil(len(t_visibles) / 7))
fig, axs = plt.subplots(filas_g, 7, figsize=(15, 2.4 * filas_g), squeeze=False)
axs = axs.reshape(-1)
for ax, t, xc, yc in zip(axs, t_visibles, col_px, fila_px):
    # Recorte de 40 x 40 px siempre dentro de la imagen (el cohete pasa pegado al borde superior)
    x0 = int(np.clip(xc - 20, 0, ANCHO - 40))
    y0 = int(np.clip(yc - 20, 0, ALTO - 40))
    ax.imshow(obtener_frame(t)[y0:y0 + 40, x0:x0 + 40], extent=[x0, x0 + 40, y0 + 40, y0])
    ax.plot(xc, yc, "o", mfc="none", mec="red", ms=14, mew=2)
    ax.set_xlim(x0, x0 + 40)
    ax.set_ylim(y0 + 40, y0)
    ax.set_title(f"t = {t:.3f} s", fontsize=8)
    ax.axis("off")
for ax in axs[len(t_visibles):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 1.4 Pasando a unidades físicas

In [ ]:
t_rel = t_visibles - t_visibles[0]
n = len(t_rel)

# Altura con escala constante (aproximacion), corrigiendo el movimiento de la camara
y_lineal = (pies_px - (fila_px - dy_cam)) * escala

# Altura con correccion de perspectiva (camara a D_cam m de distancia y h_cam m de altura)
D_cam, h_cam = 12.0, 0.0                 # (m)
f_px = D_cam / escala                    # distancia focal equivalente (px)
cy = ALTO / 2                            # fila central de la imagen
inclinacion0 = np.arctan((pies_px - cy) / f_px) - np.arctan(h_cam / D_cam)   # inclinacion de la camara en la calibracion
elevacion = inclinacion0 + np.arctan((cy - (fila_px - dy_cam)) / f_px)
y_m = h_cam + D_cam * np.tan(elevacion)  # altura sobre el suelo (m)
x_m = col_px * escala

print(f"Inclinacion de la camara en la calibracion: {np.degrees(inclinacion0):.1f} grados")
print(f"Distancia focal equivalente: {f_px:.0f} px\n")

# Tramos: grupos de cuadros consecutivos (un salto grande de tiempo abre un tramo nuevo)
paso_normal = 1.0 / FPS
nuevo_tramo = np.r_[True, np.diff(t_visibles) > 3 * paso_normal]
id_tramo = np.cumsum(nuevo_tramo)


def derivada(t, x):
    """Velocidad por diferencias centrales (en los extremos, diferencias simples)."""
    v = np.zeros(len(t))
    v[1:-1] = (x[2:] - x[:-2]) / (t[2:] - t[:-2])
    v[0] = (x[1] - x[0]) / (t[1] - t[0])
    v[-1] = (x[-1] - x[-2]) / (t[-1] - t[-2])
    return v


vx = np.full(n, np.nan)
vy = np.full(n, np.nan)
ay = np.full(n, np.nan)

for tramo in np.unique(id_tramo):
    idx = np.where(id_tramo == tramo)[0]
    if len(idx) < 2:
        continue
    vx[idx] = derivada(t_rel[idx], x_m[idx])
    vy[idx] = derivada(t_rel[idx], y_m[idx])
    if len(idx) >= 3:
        y = y_m[idx]
        ay[idx[1:-1]] = (y[2:] - 2 * y[1:-1] + y[:-2]) / paso_normal ** 2

tabla = pd.DataFrame({
    "tramo": id_tramo, "t (s)": t_rel.round(3), "x (m)": x_m.round(2), "y (m)": y_m.round(2),
    "vx (m/s)": vx.round(2), "vy (m/s)": vy.round(2), "ay (m/s2)": ay.round(1),
})
print(tabla.to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
for tramo in np.unique(id_tramo):
    idx = id_tramo == tramo
    ax[0].plot(t_rel[idx], y_m[idx], "o-", label=f"tramo {tramo}")
    ax[1].plot(t_rel[idx], vy[idx], "o-", label=f"tramo {tramo}")
ax[0].set_ylabel("y (m)")
ax[0].set_title("Altura")
ax[1].set_ylabel("vy (m/s)")
ax[1].set_title("Velocidad vertical")
for a in ax:
    a.set_xlabel("t (s)")
    a.legend(fontsize=8)
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 1.5 Exportar a Excel

In [ ]:
df_exp = pd.DataFrame({
    "t": np.round(t_rel, 4),
    "y": np.round(y_m, 3),
    "y_lineal": np.round(y_lineal, 3),
    "vy": np.round(vy, 2),
    "ay": np.round(ay, 1),
    "step": np.arange(n),
    "tramo": id_tramo,
})
df_exp.to_excel("Dato_exp_l2.xlsx", index=False)
df_exp